In [ ]:
import pandas as pd

# Load your file
df = pd.read_excel("input.xlsx")  # or csv

# Step 1: Extract family
df["Family"] = df["Material"].str.split("-").str[0]

# Step 2: Identify date columns
date_cols = [col for col in df.columns if "-" in col]

# Step 3: Group and process
result_rows = []

for family, group in df.groupby("Family"):
    
    # Sum quantities
    summed = group[date_cols].sum()
    
    # Get OE rows
    oe_rows = group[group["Segment"] == "OE"]
    
    if not oe_rows.empty:
        # Pick FIRST OE row
        oe_row = oe_rows.iloc[0]
        
        result = {
            "Material": oe_row["Material"],
            "Work Center": oe_row["Work Center"]
        }
        
        # Add summed quantities
        for col in date_cols:
            result[col] = summed[col]
        
        result_rows.append(result)

# Final DataFrame
result_df = pd.DataFrame(result_rows)

print(result_df)

In [ ]:
import pandas as pd

# Load the sheet
file_path = "D:/PPC Plan/Assembly Production 24 feb.xlsx"
df = pd.read_excel(file_path, sheet_name="23022026", header=2)

# 1. Extract Family - Using .str.split safely
df["Family"] = df["Material"].astype(str).str.split("-").str[0]

# 2. Identify date columns 
# Note: Excel dates often load as datetime objects, not strings.
# This check handles both '24-02-2026' (string) and actual datetime objects.
date_cols = [
    col for col in df.columns 
    if isinstance(col, (pd.Timestamp, datetime.date)) or ("-" in str(col) and any(char.isdigit() for char in str(col)))
]

# 3. Group and Process
result_rows = []

for family, group in df.groupby("Family"):
    # Sum the quantities for all date columns in this family
    summed_values = group[date_cols].sum(numeric_only=True)
    
    # Try to find the OE row for metadata; fallback to first row if OE doesn't exist
    oe_rows = group[group["Segment"] == "OE"]
    reference_row = oe_rows.iloc[0] if not oe_rows.empty else group.iloc[0]

    result = {
        "Family": family,
        "Material": reference_row["Material"],
        "Work Center": reference_row["WorkCenter"]
    }
    
    # Merge the summed date values into our result dictionary
    result.update(summed_values.to_dict())
    result_rows.append(result)

# Finalize and Save
result_df = pd.DataFrame(result_rows)
output_path = "D:/PPC Plan/Result_Assembly_Production.xlsx"

# Reordering columns to ensure metadata comes first
cols = ["Family", "Material", "Work Center"] + [c for c in result_df.columns if c not in ["Family", "Material", "Work Center"]]
result_df[cols].to_excel(output_path, index=False)

print(f"Result saved to {output_path}")